# TorchTitan 如何把 Metadata 喂给 Kernel

05.04 的终点是 `VarlenMetadata(cu_seq_q, cu_seq_k, max_q, max_k)`。现在的问题是：**这份 metadata 怎么从 trainer 传到 attention kernel？中间卡在哪几个配置上？**

答案的核心只有一句话：把 `inner_attention` 从默认的 `ScaledDotProductAttention.Config` 换成 `NPUVarlenAttention.Config`。但这句话背后有三件事情需要理解——

1. **为什么要换 Config 而不是换一个 flag？** 因为 TorchTitan 和 PyTorch 各有一套独立的 backend 选择机制，互不知情；
2. **换了 Config 之后发生了什么？** `NPUVarlenAttention.forward()` 把 BSND 转成 TND、把 cu_seq 转成 `actual_seq_qlen/kvlen`、调 FA v3 `sparse_mode=7`；
3. **如果只换 kernel，不声明 block-causal 语义会怎样？** trainer 不会把 packed sample 的起点转换成 metadata，kernel 也就拿不到正确的区间信息。

下面顺着这个思路走。

## 1. 决定 Attention 实现的字段：`inner_attention`

TorchTitan 的模型定义是分层的。Qwen3 的一层 Transformer block 里，`GQAttention` 负责 projection、QK norm、RoPE 和 head 映射——这些是"外层"。真正做 attention 计算的，是 `inner_attention` 这个字段指向的模块：

```
Qwen3Model.Config → layers[0] → GQAttention.Config → inner_attention
```

默认值是 `ScaledDotProductAttention.Config`。把它换成 `NPUVarlenAttention.Config`，就会替换 inner attention 模块，外层的 GQA 处理不变。完整启用 VarLen 还要同时声明 `mask_type="block_causal"`；§3 会把这两个字段接起来。

运行下面的代码直接查看当前配置：

> **要点**：`inner_attention` 决定 forward 使用哪个 attention 模块；再配合 `mask_type="block_causal"`，trainer 才会把 DataLoader 给出的样本起点转换成相应的 mask 或 metadata。接下来两节分别展开这两条线。

In [ ]:
from torchtitan_npu.models.qwen3 import model_registry

model_spec = model_registry("1.7B")
attn_config = model_spec.model.layers[0].attention
inner_config = attn_config.inner_attention

print("outer attention config:", type(attn_config).__qualname__)
print("inner attention config:", type(inner_config).__qualname__)
print("query / KV heads:", attn_config.n_heads, "/", attn_config.n_kv_heads)
print("head dim:", attn_config.head_dim)
print("mask type:", attn_config.mask_type)

inner_module = inner_config.build()
print("built inner module:", type(inner_module).__qualname__)
print("preferred SDPA backends:", inner_module.sdpa_backends)

## 2. 换了 Config 之后：`NPUVarlenAttention.forward()` 做了什么

TorchTitan 的 `Configurable` 机制让每个 Config 对象知道自己属于哪个类——调 `inner_config.build()` 就实例化出对应的模块。换 Config 对象 = 换模块实现：

| `inner_attention` 配置 | `build()` 产出 | 怎么处理 mask |
|---|---|---|
| `ScaledDotProductAttention.Config` | `ScaledDotProductAttention` | `is_causal=True` 传给 SDPA dispatcher |
| `FlexAttention.Config` | `FlexAttention` | BlockMask 描述结构化稀疏 |
| `VarlenAttention.Config` | `VarlenAttention` | cu_seq 累计长度 |
| **`NPUVarlenAttention.Config`** | **`NPUVarlenAttention`** | **cu_seq → actual_seq → FA v3 sparse_mode=7** |

`NPUVarlenAttention.forward()` 是整条链路里"翻译"发生的地方：

```text
输入: Q/K/V [B, S, N, D] in BSND + VarlenMetadata(cu_seq_q, cu_seq_k)
  → reshape BSND → TND: [B*S, N, D] 即 [T, N, D]
  → cu_seq 去掉开头的 0 → actual_seq_qlen, actual_seq_kvlen (CPU int64)
  → torch.ops.npu.npu_fusion_attention_v3(
        input_layout="TND",
        actual_seq_qlen=actual_seq,
        actual_seq_kvlen=actual_seq,
        sparse_mode=7,
        ...
    )
  → 输出 reshape 回 BSND
```

### 为什么不能只设一个 flag

那为什么不在默认 SDPA 路径里加一个 `use_varlen=True`？因为 TorchTitan 和 PyTorch 有两层独立的 backend 选择，一个 flag 管不到两层。

TorchTitan 层靠 `Configurable` 类型绑定决定实例化哪个模块——这是 §1 讲的 `inner_attention` 字段。PyTorch 层靠 dispatcher 根据 device/dtype/shape 在已注册的实现中选一个——`ScaledDotProductAttention.forward()` 里 `sdpa_kernel` 上下文的 `FLASH_ATTENTION`、`MATH` 等枚举只是候选优先级，不是执行结果。**配置里看到 `FLASH_ATTENTION` 只表示"不排斥这类 backend"——不能据此断言跑了任何特定 kernel。**

两层各管各的，一个 Python 级别的 flag 不能同时表达 TorchTitan 的模块选择与 PyTorch dispatcher 的路由规则。`NPUVarlenAttention` 仍由 TorchTitan 的 Config 类型绑定选中，但它的 `forward()` 直接调用 `npu_fusion_attention_v3`，不再经过 SDPA dispatcher。代价是只适用于 NPU；换来的是算子入口由模块实现明确决定。

> **要点**：`NPUVarlenAttention.forward()` 干了三件事——BSND→TND、cu_seq→actual_seq、调 FA v3 `sparse_mode=7`。区间边界由 `cu_seq` 传递，算子调用不经过 SDPA dispatcher。

## 3. Config 类型的双重作用

TorchTitan-NPU 的 `_enable_npu_varlen_attention(model_spec)` 做了两件事：

1. `inner_attention` → `NPUVarlenAttention.Config`
2. `mask_type` → `block_causal`

这两件事构成一个完整的握手协议。

**第一件事**把 attention 模块换成 NPU 原生实现——`NPUVarlenAttention.forward()` 负责 BSND→TND 的 layout 转换和 FA v3 的参数组装。

**第二件事**声明 packed samples 必须彼此隔离。DataLoader 已经让每条样本的位置编号从 0 重新开始；trainer 看到 `mask_type="block_causal"` 后，再根据 inner attention 的类型选择表示方式：普通 SDPA 生成 dense block-causal mask，`NPUVarlenAttention.Config` 则生成 `VarlenMetadata`。

这是整章的核心设计：DataLoader 负责保留样本起点，`mask_type` 声明隔离语义，inner-attention Config 决定 trainer 生成哪种表示以及下游如何消费。三个环节必须同时对齐。

> **要点**：`NPUVarlenAttention.Config` 选择 TND VarLen 模块；`mask_type="block_causal"` 让 trainer 读取样本起点；两者共同把正确的区间信息送到 FA v3。

<figure>
  <picture>
    <source srcset="images/05.05_torchtitan_varlen_config_flow.svg" type="image/svg+xml">
    <img src="images/05.05_torchtitan_varlen_config_flow.png" alt="TorchTitan 的 VarLen 配置流程：DataLoader 让每条样本的位置编号从 0 开始；trainer 根据 block-causal 语义找到样本起点并生成 VarlenMetadata；NPUVarlenAttention 把 BSND 与累计长度转换成 TND 和 actual_seq，最后调用 CANN FusionAttention V3 sparse_mode 7。" loading="lazy" style="max-width: 100%; height: auto;">
  </picture>
  <figcaption>图 05.05-1：<code>NPUVarlenAttention.Config</code> 是连接点——它让训练器生成的样本区间能够交给 FA v3；任一环节缺失，kernel 都不知道哪些 token 属于同一条样本。</figcaption>
</figure>

## 4. 完整链路

把 05.04 的数据路径和本节的 backend 路径接在一起。

普通 causal 路径只有两步：`SDPA + is_causal=True → PyTorch dispatcher → device kernel`。

SFT document-aware 路径则是六个环节的接力。Dataloader packing 每条样本时，都让位置编号从 0 重新开始。Trainer 收到 batch 后，看到 `mask_type="block_causal"`，便找出所有重新从 0 开始的位置；如果 inner attention 是 `NPUVarlenAttention.Config`，就用这些起点生成 flattened `VarlenMetadata`。这份 metadata 交给 `NPUVarlenAttention.forward()`——在这里 BSND 被 reshape 成 TND，cu_seq 被剥掉开头的 0 变成 `actual_seq_qlen/kvlen`，连同 `sparse_mode=7` 一起传给 CANN FA v3。最后的 kernel 在每个样本区间内部做 causal attention，区间之间互不可见。当前 wrapper 仍保留完整 `B*S` token，不应把这条链路描述成已经 compact padding。

验证这条链路需要四层同时对齐：数据层（每条样本的位置编号正确重置）、配置层（`mask_type` 与 inner Config）、调用层（实际算子参数）和执行层（profiler trace 中的 device kernel）。只看到 kernel 名称不能证明样本区间正确。

05.03 用固定模拟输入验证了最底层——`sparse_mode=7` 的语义正确性和跳算收益，同时也验证了 SDPA dispatch、FlashAttention 融合、GQA 收益边界和 document mask 语义。第 6 章把 SFT 文档边界转成 varlen metadata，跟本章建立的 baseline 做端到端对比。

## 练习

1. （判断题）只把 kernel 实现换成 `npu_fusion_attention_v3`，但不把 mask 语义设为 `block_causal`，trainer 仍会自动生成正确 metadata。

2. （单选题）当前 VarLen 配置要同时完成哪两件事？
    A. 选择 `NPUVarlenAttention` + 声明 `block_causal` 语义
    B. 加速训练 + 减小模型体积
    C. 修改 tokenizer + 调整 dataloader
    D. 切换 dtype + 调整 batch size

3. （判断题）profiler trace 里出现 VarLen kernel，只能证明算子执行了；还要检查 DataLoader 的样本起点和实际 `cu_seq`，才能证明边界正确。

4. （多选题）验证 VarLen 全链路贯通，应该检查哪些层？
    A. 数据层：每条样本的位置编号重新从 0 开始
    B. 配置层：`mask_type` 与 inner Config
    C. 调用和执行层：metadata、算子参数与 device kernel
    D. 只检查 loss 是有限值

这些拼在一起，就是 Qwen3 SFT 训练中 attention 的最终形态：packing 提高容器利用率，GQA 缩小 K/V 侧，VarLen 用 `cu_seq` + `sparse_mode=7` 隔离样本并跳过跨样本 attention 区域。GQA 管 head 维，VarLen 管 token-pair 区间，packing 管数据如何装入容器。

05.03 的固定模拟输入说明文档区间更短时，理论 pair 数和算子时间可能下降；它不等同于真实 DataLoader 的端到端加速保证。当前 wrapper 也没有在 Q/K/V 前 compact padding。第 6 章必须使用真实的样本起点、当前实现的 metadata 和重新采集的 trace，才能判断组合后的收益。

In [ ]:
!cat ./answer/05.05_answer.txt
